# NSLAB Logistics Intelligence Platform
## NSLAB-LOGISTICS-06 — ETL Silver Shanghai

**Fonte:** Cainiao-AI LaDe-P Shanghai. **Granularidade:** uma tarefa de coleta por `order_id`.

**Objetivo:** transformar o RAW imutável em Silver auditável, com tempos, classificação relativa à janela requerida e flags de qualidade. O arquivo é gravado somente após controles bloqueantes.

**Decisões do Data Owner:** `ANTES_DA_JANELA` quando coleta < início; `DENTRO_DA_JANELA` nos limites inclusivos; `APOS_A_JANELA` quando coleta > fim. Atraso apenas após o fim. Instante futuro de previsão: aceite. Nenhuma inferência de obrigação contratual.

**Premissas:** ano 2022 acrescentado a datas MM-DD (período documentado); timestamps tratados como horário local de Shanghai, sem conversão de fuso. Janela e dados cadastrais presumidos disponíveis no aceite apenas para este laboratório. GPS do aceite e `ds` **não são liberados automaticamente** para predição.

**Atenção:** as 19 colunas originais são preservadas em nome e conteúdo semântico, mas os seis campos temporais mudam de representação textual para `datetime64[ns]`. O RAW não é modificado.

**Execução:** abrir no VS Code com a `.venv` do projeto; executar sequencialmente. O Parquet RAW deve existir localmente. Não há execução do dataset neste artefato, pois o RAW não foi fornecido.
**Retificação:** ETL focado em transformações, flags, seleção justificada e carga. Distribuições exploratórias permanecem na AED. O código não foi executado com o RAW nesta revisão.


## 01 — Ambiente e caminhos

**Objetivo:** Verificar interpretador, dependências, RAW e destino, sem modificar arquivos.

**Contrato:** executar esta etapa, inspecionar a saída e só então avançar. Nenhum registro será descartado.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import display

# Identifica a raiz tanto a partir de notebooks/ quanto da raiz do projeto.
cwd = Path.cwd().resolve()
raiz = cwd.parent if cwd.name == "notebooks" else cwd
arquivo_raw = raiz / "data/raw/pickup/pickup_sh-00000-of-00001-79fabe8088e723a2.parquet"
arquivo_silver = raiz / "data/silver/pickup_shanghai_silver.parquet"

if not arquivo_raw.is_file():
    raise FileNotFoundError(f"RAW não encontrado: {arquivo_raw}")
if arquivo_raw.resolve() == arquivo_silver.resolve():
    raise RuntimeError("Origem e destino não podem coincidir.")

print("Python:", sys.version.split()[0], "| Kernel:", sys.executable)
print("Pandas:", pd.__version__, "| PyArrow:", pa.__version__)
print("RAW:", arquivo_raw, "| bytes:", arquivo_raw.stat().st_size)
print("Destino Silver:", arquivo_silver)
print("Destino já existe:", arquivo_silver.exists())

Python: 3.13.16 | Kernel: c:\Users\nslab\Documents\labs\Logistics_Intelligence_Platform\.venv\Scripts\python.exe
Pandas: 2.3.3 | PyArrow: 26.0.0
RAW: C:\Users\nslab\Documents\labs\Logistics_Intelligence_Platform\data\raw\pickup\pickup_sh-00000-of-00001-79fabe8088e723a2.parquet | bytes: 98191560
Destino Silver: C:\Users\nslab\Documents\labs\Logistics_Intelligence_Platform\data\silver\pickup_shanghai_silver.parquet
Destino já existe: False


### Explicação do código — 01 — Ambiente e caminhos

#### 1. Bibliotecas e ambiente
- `Path` manipula caminhos; `sys.executable` identifica o interpretador em uso; `pd`, `pa` e `pq` oferecem operações com tabelas e Parquet.

#### 2. Localização do projeto
- `Path.cwd().resolve()` obtém o diretório atual; a condição `cwd.name == "notebooks"` permite executar a partir da pasta de notebooks ou da raiz.

#### 3. Proteção da fonte
- `arquivo_raw.is_file()` verifica se o arquivo existe; a exceção impede continuar com um caminho incorreto.

> **Preservação:** esta etapa não altera o RAW. Os dados permanecem disponíveis para auditoria.



### Resultados, interpretação e conclusão — 01 — Ambiente e caminhos

#### 📊 Resultado
Python 3.13.16; kernel na `.venv` do projeto; Pandas 2.3.3; PyArrow 26.0.0. RAW localizado com **98.191.560 bytes**. Destino configurado em `data/silver/pickup_shanghai_silver.parquet`. No instante desta verificação, `Destino já existe: False`.

#### 🔎 Interpretação
O ambiente e a origem estavam acessíveis antes do processamento. A ausência inicial do destino não representa erro: a gravação ocorre na etapa 09.

#### ✅ Conclusão
Ambiente e caminhos validados; nenhuma alteração no RAW nesta etapa.

> **Fonte das evidências:** saídas salvas na célula de código imediatamente anterior, no notebook executado.


## 02 — Leitura e contrato estrutural

**Objetivo:** Ler o RAW, preservar a ordem das colunas e validar granularidade.

**Contrato:** executar esta etapa, inspecionar a saída e só então avançar. Nenhum registro será descartado.

In [2]:
COLUNAS_RAW = [
    "order_id", "region_id", "city", "courier_id", "accept_time",
    "time_window_start", "time_window_end", "lng", "lat", "aoi_id",
    "aoi_type", "pickup_time", "pickup_gps_time", "pickup_gps_lng",
    "pickup_gps_lat", "accept_gps_time", "accept_gps_lng", "accept_gps_lat", "ds",
]
TOTAL_AED = 1_424_406
METADADOS = pq.ParquetFile(arquivo_raw).metadata
if METADADOS.num_rows != TOTAL_AED or METADADOS.num_columns != 19:
    raise AssertionError("Metadados RAW divergentes da AED. Investigar antes de continuar.")

df_silver = pd.read_parquet(arquivo_raw, engine="pyarrow")
if list(df_silver.columns) != COLUNAS_RAW:
    raise AssertionError("Nomes/ordem das 19 colunas divergem da AED.")
if len(df_silver) != TOTAL_AED:
    raise AssertionError("Quantidade de linhas divergente da AED.")
if df_silver["order_id"].isna().any() or df_silver["order_id"].duplicated().any():
    raise AssertionError("order_id nulo ou duplicado: granularidade violada.")

# Retrato de auditoria antes das conversões.
nulos_raw = df_silver[COLUNAS_RAW].isna().sum().copy()
print("Linhas:", len(df_silver), "| Colunas RAW:", len(COLUNAS_RAW))
print("order_id único e não nulo: OK")
print("Nulos RAW registrados para controle de integridade (sem nova AED).")

Linhas: 1424406 | Colunas RAW: 19
order_id único e não nulo: OK
Nulos RAW registrados para controle de integridade (sem nova AED).


### Explicação do código — 02 — Leitura e contrato estrutural

#### 1. Contrato de origem
- `COLUNAS_RAW` define explicitamente as 19 colunas; `TOTAL_AED` registra o total histórico esperado.

#### 2. Inspeção e leitura
- `pq.ParquetFile(...).metadata` consulta metadados; `pd.read_parquet()` carrega os registros para transformação em memória.

#### 3. Granularidade e ausências
- `duplicated("order_id")` verifica repetição da chave; `isna().sum()` registra ausências por coluna antes do tratamento.

> **Preservação:** esta etapa não altera o RAW. Os dados permanecem disponíveis para auditoria.



### Resultados, interpretação e conclusão — 02 — Leitura e contrato estrutural

#### 📊 Resultado
RAW lido com **1.424.406 linhas e 19 colunas**. Verificação de `order_id` único e não nulo: **OK**. Nulos originais registrados para controle de integridade.

#### 🔎 Interpretação
O contrato estrutural e a chave foram confirmados sem descartar registros. O registro de nulos é controle de origem, não imputação.

#### ✅ Conclusão
Leitura e granularidade aprovadas para prosseguir às transformações.

> **Fonte das evidências:** saídas salvas na célula de código imediatamente anterior, no notebook executado.


## 03 — Conversão temporal

**Objetivo:** Converter seis timestamps com ano técnico 2022 e auditar falhas.

**Contrato:** executar esta etapa, inspecionar a saída e só então avançar. Nenhum registro será descartado.

In [3]:
COLUNAS_TEMPO = [
    "accept_time", "pickup_time", "time_window_start", "time_window_end",
    "accept_gps_time", "pickup_gps_time",
]
diagnostico = []
for coluna in COLUNAS_TEMPO:
    antes = df_silver[coluna]
    texto = antes.astype("string").str.strip()
    convertido = pd.to_datetime("2022-" + texto, format="%Y-%m-%d %H:%M:%S", errors="coerce")
    falhas = int((antes.notna() & convertido.isna()).sum())
    diagnostico.append({"coluna": coluna, "nulos_raw": int(antes.isna().sum()),
                        "falhas": falhas, "nulos_silver": int(convertido.isna().sum())})
    df_silver[coluna] = convertido

df_tempos = pd.DataFrame(diagnostico)
display(df_tempos)
if df_tempos["falhas"].sum() != 0:
    raise AssertionError("Falha de parsing temporal: investigar antes de prosseguir.")
if not (df_tempos["nulos_raw"] == df_tempos["nulos_silver"]).all():
    raise AssertionError("Mudança inesperada na quantidade de nulos temporais.")
if not all(pd.api.types.is_datetime64_ns_dtype(df_silver[c]) for c in COLUNAS_TEMPO):
    raise TypeError("Conversão temporal não produziu datetime64[ns].")
print("Conversão dos seis campos: OK")

,coluna,nulos_raw,falhas,nulos_silver
0,accept_time,0,0,0
1,pickup_time,0,0,0
2,time_window_start,0,0,0
3,time_window_end,0,0,0
4,accept_gps_time,654425,0,654425
5,pickup_gps_time,441333,0,441333


Conversão dos seis campos: OK


### Explicação do código — 03 — Conversão temporal

#### 1. Seleção dos campos temporais
- `COLUNAS_TEMPO` enumera os seis campos a converter, evitando transformações implícitas em outras colunas.

#### 2. Conversão explícita
- `astype("string").str.strip()` normaliza espaços; `pd.to_datetime(..., format=..., errors="coerce")` converte para data e hora, sinalizando entradas inválidas como `NaT`.

#### 3. Auditoria de falhas
- `antes.notna() & convertido.isna()` identifica valores existentes que não puderam ser convertidos; o diagnóstico separa nulos de origem e falhas novas.

> **Preservação:** esta etapa não altera o RAW. Os dados permanecem disponíveis para auditoria.



### Resultados, interpretação e conclusão — 03 — Conversão temporal

#### 📊 Resultado
Os seis campos temporais foram convertidos com **zero falhas**. `accept_time`, `pickup_time`, `time_window_start` e `time_window_end`: zero nulos antes e depois. `accept_gps_time`: **654.425** nulos preservados; `pickup_gps_time`: **441.333** nulos preservados. Ano técnico adotado: **2022**.

#### 🔎 Interpretação
A conversão não criou nulos adicionais e preservou as ausências já presentes nos campos GPS. O ano de 2022 decorre da premissa documentada do laboratório, pois o ano não está explícito nos timestamps de origem.

#### ✅ Conclusão
Conversão temporal validada, sem imputação e sem exclusão de linhas.

> **Fonte das evidências:** saídas salvas na célula de código imediatamente anterior, no notebook executado.


## 04 — Métricas operacionais

**Objetivo:** Calcular durações e desvios assinados, sem alterar registros.

**Contrato:** executar esta etapa, inspecionar a saída e só então avançar. Nenhum registro será descartado.

In [4]:
def minutos(fim, inicio):
    """Diferença temporal vetorizada em minutos; preserva NaN."""
    return (df_silver[fim] - df_silver[inicio]).dt.total_seconds() / 60.0

df_silver["janela_min"] = minutos("time_window_end", "time_window_start")
df_silver["aceite_ate_coleta_min"] = minutos("pickup_time", "accept_time")
df_silver["aceite_ate_inicio_janela_min"] = minutos("time_window_start", "accept_time")
df_silver["aceite_ate_fim_janela_min"] = minutos("time_window_end", "accept_time")
df_silver["desvio_fim_janela_min"] = minutos("pickup_time", "time_window_end")
df_silver["minutos_apos_limite"] = df_silver["desvio_fim_janela_min"].clip(lower=0)
df_silver["minutos_antes_inicio"] = minutos("time_window_start", "pickup_time").clip(lower=0)

METRICAS = ["janela_min", "aceite_ate_coleta_min", "aceite_ate_inicio_janela_min",
            "aceite_ate_fim_janela_min", "desvio_fim_janela_min",
            "minutos_apos_limite", "minutos_antes_inicio"]
print("Métricas derivadas calculadas:", len(METRICAS))
print("Métricas criadas:", METRICAS)

Métricas derivadas calculadas: 7
Métricas criadas: ['janela_min', 'aceite_ate_coleta_min', 'aceite_ate_inicio_janela_min', 'aceite_ate_fim_janela_min', 'desvio_fim_janela_min', 'minutos_apos_limite', 'minutos_antes_inicio']


### Explicação do código — 04 — Métricas operacionais

#### 1. Função reutilizável
- `minutos(fim, inicio)` subtrai dois campos datetime e converte a diferença com `.dt.total_seconds() / 60.0`.

#### 2. Métricas operacionais
- `janela_min` mede a duração requerida; `aceite_ate_coleta_min` mede o tempo observado; as diferenças entre aceite e limites da janela preservam sinais.

#### 3. Desvios e limites
- `desvio_fim_janela_min` mantém o sinal da diferença; `.clip(lower=0)` isola minutos positivos após o limite, sem transformar valores ausentes em zero.

> **Preservação:** esta etapa não altera o RAW. Os dados permanecem disponíveis para auditoria.



### Resultados, interpretação e conclusão — 04 — Métricas operacionais

#### 📊 Resultado
**Sete métricas derivadas calculadas:** `janela_min`, `aceite_ate_coleta_min`, `aceite_ate_inicio_janela_min`, `aceite_ate_fim_janela_min`, `desvio_fim_janela_min`, `minutos_apos_limite` e `minutos_antes_inicio`.

#### 🔎 Interpretação
As diferenças temporais permitem analisar duração da janela, intervalo entre aceite e coleta e desvios relativos aos limites. O resultado exibido comprova a criação das sete colunas, mas não apresenta estatísticas descritivas individuais nesta etapa.

#### ✅ Conclusão
Métricas materializadas; valores e regras históricas são conferidos na reconciliação da etapa 07.

> **Fonte das evidências:** saídas salvas na célula de código imediatamente anterior, no notebook executado.


## 05 — Classificação da janela

**Objetivo:** Aplicar as três categorias aprovadas e a flag de atraso.

**Contrato:** executar esta etapa, inspecionar a saída e só então avançar. Nenhum registro será descartado.

In [5]:
inicio = df_silver["time_window_start"]
fim = df_silver["time_window_end"]
coleta = df_silver["pickup_time"]
validos = inicio.notna() & fim.notna() & coleta.notna() & inicio.le(fim)

# Categoria desconhecida para entradas incompletas ou janela invertida.
classe = pd.Series(pd.NA, index=df_silver.index, dtype="string")
classe.loc[validos & coleta.lt(inicio)] = "ANTES_DA_JANELA"
classe.loc[validos & coleta.ge(inicio) & coleta.le(fim)] = "DENTRO_DA_JANELA"
classe.loc[validos & coleta.gt(fim)] = "APOS_A_JANELA"
df_silver["classificacao_janela"] = classe.astype("category")

# Flag de atraso exclusivamente por pickup_time > time_window_end.
# Não confundir com a classificação, que também requer janela válida.
df_silver["coleta_apos_limite"] = coleta.gt(fim).where(coleta.notna() & fim.notna()).astype("boolean")
print("Categorias (incluindo ausentes):")
display(df_silver["classificacao_janela"].value_counts(dropna=False).to_frame("registros"))
print("Coletas após o limite:", int(df_silver["coleta_apos_limite"].sum()))

Categorias (incluindo ausentes):


,registros
classificacao_janela,
DENTRO_DA_JANELA,1027561
ANTES_DA_JANELA,360315
APOS_A_JANELA,36530


Coletas após o limite: 36530


### Explicação do código — 05 — Classificação da janela

#### 1. Pré-condição da classificação
- `notna()` exige datas presentes; `inicio.le(fim)` impede classificar automaticamente janelas invertidas.

#### 2. Três categorias aprovadas
- `.lt(inicio)` representa antes; `.ge(inicio) & .le(fim)` representa dentro, incluindo os limites; `.gt(fim)` representa após.

#### 3. Indicador de atraso
- `coleta_apos_limite` compara coleta e fim da janela; não equivale a uma obrigação contratual. Categorias indefinidas permanecem auditáveis.

> **Preservação:** esta etapa não altera o RAW. Os dados permanecem disponíveis para auditoria.



### Resultados, interpretação e conclusão — 05 — Classificação da janela

#### 📊 Resultado
`DENTRO_DA_JANELA`: **1.027.561** (72,14%); `ANTES_DA_JANELA`: **360.315** (25,30%); `APOS_A_JANELA`: **36.530** (2,56%). Total: **1.424.406**. Coletas após o limite: **36.530**.

#### 🔎 Interpretação
A maioria das coletas ocorreu dentro da janela requerida. Coleta antecipada não é classificada automaticamente como falha. O atraso é definido exclusivamente por `pickup_time > time_window_end`, sem alegar SLA contratual. As três contagens somam o total de registros.

#### ✅ Conclusão
Classificação materializada e contagem de atrasos consistente com a AED.

> **Fonte das evidências:** saídas salvas na célula de código imediatamente anterior, no notebook executado.


## 06 — Indicadores de qualidade

**Objetivo:** Sinalizar incoerências temporais, ausência de GPS e coordenadas fisicamente inválidas.

**Contrato:** executar esta etapa, inspecionar a saída e só então avançar. Nenhum registro será descartado.

In [6]:
df_silver["flag_temporal_essencial_ausente"] = df_silver[
    ["accept_time", "pickup_time", "time_window_start", "time_window_end"]
].isna().any(axis=1)
df_silver["flag_janela_invertida"] = df_silver["janela_min"].lt(0)
df_silver["flag_coleta_antes_aceite"] = df_silver["aceite_ate_coleta_min"].lt(0)
df_silver["flag_aceite_apos_fim_janela"] = minutos("accept_time", "time_window_end").gt(0)
df_silver["flag_coleta_antes_inicio_janela"] = coleta.lt(inicio)

PARES_GPS = {"aceite": ("accept_gps_lng", "accept_gps_lat", "accept_gps_time"),
             "coleta": ("pickup_gps_lng", "pickup_gps_lat", "pickup_gps_time")}
for evento, (lon, lat, instante) in PARES_GPS.items():
    df_silver[f"flag_gps_{evento}_ausente"] = df_silver[[lon, lat, instante]].isna().any(axis=1)
    df_silver[f"flag_gps_{evento}_parcial"] = df_silver[[lon, lat, instante]].isna().any(axis=1) & df_silver[[lon, lat, instante]].notna().any(axis=1)

for nome, lon, lat in [("referencia", "lng", "lat"),
                       ("aceite", "accept_gps_lng", "accept_gps_lat"),
                       ("coleta", "pickup_gps_lng", "pickup_gps_lat")]:
    x = pd.to_numeric(df_silver[lon], errors="coerce")
    y = pd.to_numeric(df_silver[lat], errors="coerce")
    df_silver[f"flag_coord_{nome}_fora_limites"] = (
        (x.notna() & ~x.between(-180, 180)) | (y.notna() & ~y.between(-90, 90))
    )

# ds é MMDD de pickup_time: diagnóstico, não preditor no aceite.
mmdd = df_silver["pickup_time"].dt.month * 100 + df_silver["pickup_time"].dt.day
df_silver["flag_ds_divergente"] = df_silver["ds"].ne(mmdd).where(mmdd.notna() & df_silver["ds"].notna()).astype("boolean")
FLAGS = [c for c in df_silver.columns if c.startswith("flag_")]
print("Indicadores de qualidade materializados:", len(FLAGS))

Indicadores de qualidade materializados: 13


### Explicação do código — 06 — Indicadores de qualidade

#### 1. Consistência temporal
- `isna().any(axis=1)` identifica linhas sem datas essenciais; `.lt(0)` e `.gt(0)` sinalizam cronologias potencialmente incoerentes.

#### 2. Integridade geográfica
- As flags GPS registram ausências e coordenadas fora de limites globais, sem afirmar que distâncias atípicas sejam necessariamente erros.

#### 3. Auditoria de data
- A comparação de `ds` com mês/dia de `pickup_time` verifica consistência documental, sem liberar `ds` como preditor no aceite.

> **Preservação:** esta etapa não altera o RAW. Os dados permanecem disponíveis para auditoria.



### Resultados, interpretação e conclusão — 06 — Indicadores de qualidade

#### 📊 Resultado
**13 indicadores de qualidade materializados**, conforme a mensagem de execução.

#### 🔎 Interpretação
As flags permitem identificar e auditar condições de qualidade sem apagar ou corrigir automaticamente os registros. A saída desta célula não informa as contagens individuais de cada flag; portanto, não se atribuem números não observados.

#### ✅ Conclusão
Indicadores criados; qualquer investigação de ocorrências específicas deve usar os dados e verificações correspondentes.

> **Fonte das evidências:** saídas salvas na célula de código imediatamente anterior, no notebook executado.


## 07 — Reconciliação com a AED

**Objetivo:** Confrontar apenas indicadores históricos documentados e bloquear divergências.

**Contrato:** executar esta etapa, inspecionar a saída e só então avançar. Nenhum registro será descartado.

In [7]:
esperados = {
    "registros": 1_424_406,
    "coletas_apos_limite": 36_530,
    "mediana_janela_min": 120.0,
    "mediana_aceite_ate_coleta_min": 124.0,
    "mediana_desvio_fim_janela_min": -88.0,
    "gps_coleta_ausente": 441_333,
    "gps_aceite_ausente": 654_425,
    "ds_divergente": 0,
}
observados = {
    "registros": len(df_silver),
    "coletas_apos_limite": int(df_silver["coleta_apos_limite"].sum()),
    "mediana_janela_min": float(df_silver["janela_min"].median()),
    "mediana_aceite_ate_coleta_min": float(df_silver["aceite_ate_coleta_min"].median()),
    "mediana_desvio_fim_janela_min": float(df_silver["desvio_fim_janela_min"].median()),
    "gps_coleta_ausente": int(df_silver["pickup_gps_time"].isna().sum()),
    "gps_aceite_ausente": int(df_silver["accept_gps_time"].isna().sum()),
    "ds_divergente": int(df_silver["flag_ds_divergente"].sum()),
}
comparacao_aed = pd.DataFrame([
    {"indicador": k, "AED": v, "Silver": observados[k],
     "confere": abs(observados[k] - v) <= (0.5 if k.startswith("mediana_") else 0)}
    for k, v in esperados.items()
])
display(comparacao_aed)
if not comparacao_aed["confere"].all():
    raise AssertionError("Divergência com AED: investigar antes de gravar Silver.")
print("Reconciliação AED: OK")

,indicador,AED,Silver,confere
0,registros,1424406.0,1424406.0,True
1,coletas_apos_limite,36530.0,36530.0,True
2,mediana_janela_min,120.0,120.0,True
3,mediana_aceite_ate_coleta_min,124.0,124.0,True
4,mediana_desvio_fim_janela_min,-88.0,-88.0,True
5,gps_coleta_ausente,441333.0,441333.0,True
6,gps_aceite_ausente,654425.0,654425.0,True
7,ds_divergente,0.0,0.0,True


Reconciliação AED: OK


### Explicação do código — 07 — Reconciliação com a AED

#### 1. Referências históricas
- `esperados` guarda os indicadores consolidados da AED; `observados` calcula os mesmos indicadores a partir da Silver em memória.

#### 2. Reconciliação quantitativa
- A tabela `comparacao_aed` expõe valor esperado, observado e concordância, tornando divergências inspecionáveis.

#### 3. Controle bloqueante
- A verificação final interrompe o fluxo diante de discrepâncias; não corrige valores nem altera regras apenas para reproduzir a AED.

> **Preservação:** esta etapa não altera o RAW. Os dados permanecem disponíveis para auditoria.



### Resultados, interpretação e conclusão — 07 — Reconciliação com a AED

#### 📊 Resultado
**Oito indicadores conferidos; todos `True`**: registros **1.424.406**; coletas após o limite **36.530**; mediana da janela **120 min**; mediana entre aceite e coleta **124 min**; mediana do desvio ao fim da janela **−88 min**; GPS da coleta ausente **441.333**; GPS do aceite ausente **654.425**; divergências de `ds` **0**. Mensagem final: **`Reconciliação AED: OK`**.

#### 🔎 Interpretação
As oito métricas calculadas na Silver coincidem com as referências históricas da AED, fornecendo controle de regressão para as transformações. Isso não elimina outras limitações semânticas documentadas.

#### ✅ Conclusão
Reconciliação aprovada para os oito indicadores testados.

> **Fonte das evidências:** saídas salvas na célula de código imediatamente anterior, no notebook executado.


## 07B — Seleção justificada de colunas Silver

**Objetivo:** explicitar o contrato de colunas após a transformação. A exclusão de colunas é permitida apenas com justificativa documentada; não existe, nesta revisão, evidência suficiente para excluir automaticamente nenhuma das 19 originais. `city`, `ds` e GPS permanecem para auditoria e possíveis usos analíticos, embora alguns sejam inadequados como preditores no aceite.


In [8]:
# Contrato explícito: não excluir automaticamente variáveis originais.
COLUNAS_DESCARTADAS = {}  # coluna -> justificativa aprovada e documentada
if COLUNAS_DESCARTADAS:
    raise RuntimeError("Descarte proposto exige revisão documentada do contrato DO-03 antes de executar.")
COLUNAS_SILVER = COLUNAS_RAW + [c for c in df_silver.columns if c not in COLUNAS_RAW]
df_silver = df_silver.loc[:, COLUNAS_SILVER].copy()
print("Colunas originais mantidas:", len(COLUNAS_RAW))
print("Colunas derivadas:", len(COLUNAS_SILVER) - len(COLUNAS_RAW))
print("Colunas descartadas:", len(COLUNAS_DESCARTADAS))


Colunas originais mantidas: 19
Colunas derivadas: 22
Colunas descartadas: 0


### Explicação do código — 07B — Seleção justificada

#### 1. Política de descarte
- `COLUNAS_DESCARTADAS` registra o vínculo entre uma coluna e sua justificativa aprovada.
- O bloqueio com `raise RuntimeError` impede aplicar exclusões sem revisão da decisão DO-03.

#### 2. Organização do esquema
- `COLUNAS_RAW + [...]` coloca as colunas de origem primeiro e acrescenta as derivadas sem duplicação.
- `.loc[:, COLUNAS_SILVER].copy()` materializa a ordem final do DataFrame.

#### 3. Rastreabilidade
- `print()` apresenta somente contagens objetivas do contrato, sem reabrir investigações da AED.


### Resultados, interpretação e conclusão — 07B — Seleção justificada de colunas Silver

#### 📊 Resultado
**19 colunas originais mantidas; 22 derivadas; zero descartadas**.

#### 🔎 Interpretação
A política de preservação mantém rastreabilidade e permite auditorias posteriores. A ausência de descarte não significa que todas as colunas serão usadas como features de Machine Learning.

#### ✅ Conclusão
Contrato de colunas da Silver aprovado nesta execução, sem exclusões.

> **Fonte das evidências:** saídas salvas na célula de código imediatamente anterior, no notebook executado.


## 08 — Integridade final e separação temporal

**Objetivo:** Garantir preservação das colunas e documentar disponibilidade no aceite.

**Contrato:** executar esta etapa, inspecionar a saída e só então avançar. Nenhum registro será descartado.

In [9]:
if len(df_silver) != TOTAL_AED or not set(COLUNAS_RAW).issubset(df_silver.columns):
    raise AssertionError("Linhas ou ordem das 19 colunas RAW alteradas.")
if df_silver["order_id"].isna().any() or df_silver["order_id"].duplicated().any():
    raise AssertionError("Chave order_id inválida.")
if not df_silver[COLUNAS_RAW].isna().sum().eq(nulos_raw).all():
    raise AssertionError("Mudança de nulos nas colunas de origem.")
if not df_silver["coleta_apos_limite"].eq(df_silver["desvio_fim_janela_min"].gt(0)).where(df_silver["coleta_apos_limite"].notna(), True).all():
    raise AssertionError("Flag de atraso incompatível com o desvio.")

# Lista documental, NÃO usada para criar um conjunto de treino nesta fase.
COLUNAS_POS_ACEITE = ["pickup_time", "pickup_gps_time", "pickup_gps_lng", "pickup_gps_lat",
                     "ds", "classificacao_janela", "coleta_apos_limite",
                     "aceite_ate_coleta_min", "desvio_fim_janela_min",
                     "minutos_apos_limite", "minutos_antes_inicio"]
COLUNAS_DISPONIBILIDADE_INCERTA = ["accept_gps_time", "accept_gps_lng", "accept_gps_lat"]
print("Integridade: OK | linhas:", len(df_silver), "| colunas:", len(df_silver.columns))
print("Campos pós-aceite / desfecho (não usar como preditores):", COLUNAS_POS_ACEITE)
print("Disponibilidade não confirmada:", COLUNAS_DISPONIBILIDADE_INCERTA)
print("Atenção: outras flags e métricas derivadas de eventos futuros também são vedadas como preditores.")

Integridade: OK | linhas: 1424406 | colunas: 41
Campos pós-aceite / desfecho (não usar como preditores): ['pickup_time', 'pickup_gps_time', 'pickup_gps_lng', 'pickup_gps_lat', 'ds', 'classificacao_janela', 'coleta_apos_limite', 'aceite_ate_coleta_min', 'desvio_fim_janela_min', 'minutos_apos_limite', 'minutos_antes_inicio']
Disponibilidade não confirmada: ['accept_gps_time', 'accept_gps_lng', 'accept_gps_lat']
Atenção: outras flags e métricas derivadas de eventos futuros também são vedadas como preditores.


### Explicação do código — 08 — Integridade final e separação temporal

#### 1. Preservação estrutural
- `len(df_silver)` verifica linhas; `df_silver.columns[:19]` verifica que as 19 colunas originais continuam na ordem prevista.

#### 2. Integridade da chave e dos nulos
- `isna().any()` e `duplicated().any()` verificam `order_id`; `eq(nulos_raw).all()` compara contagens de ausências com a origem.

#### 3. Disponibilidade no aceite
- As listas de campos documentam possíveis vazamentos: eventos de coleta e métricas de desfecho não são preditores disponíveis no instante de aceite.

> **Preservação:** esta etapa não altera o RAW. Os dados permanecem disponíveis para auditoria.



### Resultados, interpretação e conclusão — 08 — Integridade final e separação temporal

#### 📊 Resultado
**`Integridade: OK`**; **1.424.406 linhas e 41 colunas**. O código sinalizou como pós-aceite/desfecho: `pickup_time`, `pickup_gps_time`, `pickup_gps_lng`, `pickup_gps_lat`, `ds`, `classificacao_janela`, `coleta_apos_limite`, `aceite_ate_coleta_min`, `desvio_fim_janela_min`, `minutos_apos_limite`, `minutos_antes_inicio`. Disponibilidade não confirmada: `accept_gps_time`, `accept_gps_lng`, `accept_gps_lat`.

#### 🔎 Interpretação
A estrutura da Silver está preservada. Os campos de desfecho e seus derivados não podem alimentar um modelo que prevê no momento do aceite; o próprio código alerta que outras flags e métricas derivadas de eventos futuros também devem ser excluídas dos preditores. A disponibilidade do GPS do aceite permanece pendente de confirmação.

#### ✅ Conclusão
Integridade aprovada; seleção definitiva de features fica para a etapa de modelagem, com controle explícito de *data leakage*.

> **Fonte das evidências:** saídas salvas na célula de código imediatamente anterior, no notebook executado.


## 09 — Persistência controlada em Parquet

**Objetivo:** Gravar somente após aprovação explícita no notebook, com compressão Snappy.

**Contrato:** executar esta etapa, inspecionar a saída e só então avançar. Nenhum registro será descartado.

In [12]:
# Segurança: mudar para True APENAS após conferir os resultados anteriores.
AUTORIZAR_GRAVACAO = True  # Alterar para True somente após validar as etapas anteriores

if not AUTORIZAR_GRAVACAO:
    print("Gravação NÃO autorizada. Revise as validações e altere AUTORIZAR_GRAVACAO para True.")
else:
    if not comparacao_aed["confere"].all():
        raise AssertionError("Reconciliação não aprovada.")
    if arquivo_silver.exists():
        raise FileExistsError(f"Silver já existe; não sobrescrever automaticamente: {arquivo_silver}")
    arquivo_silver.parent.mkdir(parents=True, exist_ok=True)
    # Grava primeiro em arquivo temporário no mesmo diretório.
    temporario = arquivo_silver.with_name(arquivo_silver.name + ".tmp")
    if temporario.exists():
        raise FileExistsError(f"Temporário já existe: {temporario}")
    try:
        df_silver.to_parquet(temporario, engine="pyarrow", compression="snappy", index=False)
        # Metadados básicos são verificados antes da publicação.
        meta_tmp = pq.ParquetFile(temporario).metadata
        if meta_tmp.num_rows != TOTAL_AED or meta_tmp.num_columns != len(df_silver.columns):
            raise AssertionError("Parquet temporário diverge do DataFrame.")
        temporario.replace(arquivo_silver)
    finally:
        # Remove apenas temporário criado por esta etapa, nunca o RAW.
        if temporario.exists():
            temporario.unlink()
    print("Silver gravada:", arquivo_silver)
    print("Tamanho (bytes):", arquivo_silver.stat().st_size)

Silver gravada: C:\Users\nslab\Documents\labs\Logistics_Intelligence_Platform\data\silver\pickup_shanghai_silver.parquet
Tamanho (bytes): 56461223


### Explicação do código — 09 — Persistência controlada em Parquet

#### 1. Autorização explícita
- A configuração segura inicial é `AUTORIZAR_GRAVACAO = False`, que mantém a persistência desligada até revisão manual. **Nesta execução registrada, o parâmetro foi alterado para `True`**, autorizando a gravação após as validações.

#### 2. Proteção contra sobrescrita
- `arquivo_silver.exists()` impede substituir automaticamente uma Silver existente; a reconciliação da AED é reavaliada antes de gravar.

#### 3. Escrita e publicação
- `to_parquet(index=False, compression="snappy")` grava sem índice artificial; o arquivo temporário é conferido antes de ser publicado no destino.

> **Preservação:** esta etapa não altera o RAW. A escrita da Silver somente ocorre mediante autorização explícita.



### Resultados, interpretação e conclusão — 09 — Persistência controlada em Parquet

#### 📊 Resultado
`AUTORIZAR_GRAVACAO = True` na execução registrada. Arquivo gravado em `data/silver/pickup_shanghai_silver.parquet`, com **56.461.223 bytes**.

#### 🔎 Interpretação
A gravação foi explicitamente autorizada nesta execução, após as verificações anteriores. A mensagem confirma a escrita do artefato; a integridade após releitura é tratada separadamente na etapa 10.

#### ✅ Conclusão
Persistência da Silver concluída na execução documentada.

> **Fonte das evidências:** saídas salvas na célula de código imediatamente anterior, no notebook executado.


## 10 — Releitura e validação física

**Objetivo:** Conferir arquivo Silver gravado, schema, chave e métricas após round-trip.

**Contrato:** executar esta etapa, inspecionar a saída e só então avançar. Nenhum registro será descartado.

In [13]:
if not arquivo_silver.exists():
    print("Silver ainda não gravada. Execute a etapa 09 com autorização após validação.")
else:
    meta_silver = pq.ParquetFile(arquivo_silver).metadata
    df_releitura = pd.read_parquet(arquivo_silver, engine="pyarrow")
    assert meta_silver.num_rows == TOTAL_AED
    assert meta_silver.num_columns == len(df_silver.columns)
    assert df_releitura.columns.tolist() == df_silver.columns.tolist()
    assert df_releitura["order_id"].is_unique
    assert int(df_releitura["coleta_apos_limite"].sum()) == 36_530
    assert df_releitura["classificacao_janela"].value_counts(dropna=False).sum() == TOTAL_AED
    assert df_releitura[COLUNAS_RAW].isna().sum().eq(nulos_raw).all()
    assert all(pd.api.types.is_datetime64_ns_dtype(df_releitura[c]) for c in COLUNAS_TEMPO)
    print("Validação física e lógica após releitura: OK")
    print("Linhas:", len(df_releitura), "| Colunas:", len(df_releitura.columns))
    display(df_releitura[["order_id", "classificacao_janela", "minutos_apos_limite"]].head())

Validação física e lógica após releitura: OK
Linhas: 1424406 | Colunas: 41


,order_id,classificacao_janela,minutos_apos_limite
0,2349637,DENTRO_DA_JANELA,0.0
1,4867696,DENTRO_DA_JANELA,0.0
2,5691514,DENTRO_DA_JANELA,0.0
3,1443776,ANTES_DA_JANELA,0.0
4,1806717,DENTRO_DA_JANELA,0.0


### Explicação do código — 10 — Releitura e validação física

#### 1. Verificação do artefato
- `arquivo_silver.exists()` diferencia ausência de arquivo de falha de leitura; `pq.ParquetFile(...).metadata` consulta dimensões persistidas.

#### 2. Round-trip
- `pd.read_parquet()` recarrega o arquivo; comparar colunas, linhas, unicidade e métricas detecta diferenças entre memória e disco.

#### 3. Critério de aceite
- `assert` interrompe a etapa se a integridade física falhar; a execução bem-sucedida deve ser documentada com evidências reais.

> **Preservação:** esta etapa não altera o RAW. Os dados permanecem disponíveis para auditoria.



### Resultados, interpretação e conclusão — 10 — Releitura e validação física

#### 📊 Resultado
Mensagem registrada: **`Validação física e lógica após releitura: OK`**. Arquivo relido com **1.424.406 linhas e 41 colunas**. A amostra apresentada contém `order_id`, `classificacao_janela` e `minutos_apos_limite`, incluindo registros dentro e antes da janela.

#### 🔎 Interpretação
A releitura confirma que o arquivo persistido pode ser aberto e mantém as dimensões esperadas. A amostra demonstra a presença de colunas essenciais, mas não substitui auditorias adicionais que possam ser exigidas em produção.

#### ✅ Conclusão
Validação física e lógica registrada como aprovada no notebook; a Silver está pronta para documentação e etapas seguintes do laboratório.

> **Fonte das evidências:** saídas salvas na célula de código imediatamente anterior, no notebook executado.


## Encerramento e critérios de aceite

O ETL estará concluído **somente** quando: (1) etapas 01–08 aprovadas; (2) gravação explicitamente autorizada; (3) etapa 10 aprovada; (4) resultados reais documentados nos espaços de cada etapa; (5) Git status revisado antes de commit/push.

**Fora do escopo:** Gold, treinamento de modelos, classificação preditiva, regressão, clustering e otimização de rotas.

**Rastreabilidade:** fonte RAW preservada; ano 2022 inferido do período documentado; todas as 19 colunas originais preservadas; indicadores adicionais identificados por nomes próprios.

**Observação metodológica:** inconsistências temporais não provocam exclusão automática. A categoria de janela é nula quando os limites estão ausentes ou invertidos; a flag de atraso continua refletindo estritamente a comparação com o fim da janela quando ambos os tempos existem.

In [14]:
df_silver.dtypes

order_id                                       int64
region_id                                      int64
city                                          object
courier_id                                     int64
accept_time                           datetime64[ns]
time_window_start                     datetime64[ns]
time_window_end                       datetime64[ns]
lng                                          float64
lat                                          float64
aoi_id                                         int64
aoi_type                                       int64
pickup_time                           datetime64[ns]
pickup_gps_time                       datetime64[ns]
pickup_gps_lng                               float64
pickup_gps_lat                               float64
accept_gps_time                       datetime64[ns]
accept_gps_lng                               float64
accept_gps_lat                               float64
ds                                            